Ce notebook a été utilisé pour mosaïquer, en moyennant les recouvrements, les images L3A S2 utilisées pour le stage.
La première version de ce traitement utilisait la librairie odc-geo pour paralléliser les reprojections, celle-ci utilise simplement la reprojection rasterio.

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
import geopandas as gpd
import shreklib
import os
from os.path import join
import pandas as pd
import glob
import matplotlib.pyplot as plt
from affine import Affine
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [ ]:
# fonction de reprojection à la grille de référence du stage
def reproj_to_ref(in_fp):

    out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/Sentinel2/Haute-Garonne/l3a_snapped"
    ref_transform=Affine(*shreklib.ref_grid["transform"].ravel()[:6])

    out_fp=join(out_folder, os.path.basename(in_fp))
    profile={
        "crs":32631,
        "transform":ref_transform,
        "width":shreklib.ref_grid["width"],
        "height":shreklib.ref_grid["height"],
        "nodata":-32768,
        "dtype":"int16", 
        "count":1
    }
    with rasterio.open(in_fp, "r") as src:
        with rasterio.open(out_fp, "w", **profile) as dst:
            reproject(
                source=rasterio.band(src,1), #rasterio.band donne directement les instructions à gdal et ne charge pas tout en mémoire python.
                destination=rasterio.band(dst,1),

                src_transform=src.transform,
                dst_transform=ref_transform,

                src_crs=src.crs,
                dst_crs=32631,

                dst_nodata=-32768,
                resampling=Resampling.bilinear
            )

In [2]:
#chemin vers un dossier contennant les L3A téléchargées via GEODES et dézippées
folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/Sentinel2/Haute-Garonne/GEODES/unzipped"

#chemin vers une image aux crs, dimensions et transformation du projet, pour reprojetter les images à la même grille
ref_grid_img_fps="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/backscatter_stats/ascending_hh.tif"

In [ ]:
with rasterio.open(ref_grid_img_fps, "r") as src:
    ref_transform=src.transform
    ref_height=src.height
    ref_width=src.width
    ref_crs=src.crs
ref_shape=(ref_height, ref_width)
fps=glob.glob(join(folder, "**/*.tif"), recursive=True)
# on filtre uniquement les synthèses de version 4
V4_mask=list(map(lambda fp:os.path.basename(fp).split("_")[-3][:2]=="V4", fps))
v4_fps=np.array(fps)[V4_mask]
in_fp=fps[0]
#reprojection parallélisée
shreklib.batch_map(reproj_to_ref, v4_fps)

# création des moyennes mensuelles à partir des images reprojettées

In [ ]:
def load(fp):
    with rasterio.open(fp, "r") as src:
        return src.read(1)

#fonction pour mosaïquer les images en moyennant les recouvrements et les exporter en .tif
def mosaic(group):
    out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/Sentinel2/Haute-Garonne/month_bands_mosaics"
    imgs=[]
    for fp in group[1].fps:
        imgs.append(load(fp))
    aggregate=np.zeros(ref_shape, dtype=np.int64)
    valid_count=np.zeros(ref_shape, dtype=np.uint8)
    nodata=-32768
    for img in imgs:
        mask=img!=nodata
        valid_count+=mask.astype(np.uint8)
        np.add(aggregate, img, out=aggregate, where=mask)
    month, band=group[0]
    outname=f"month{str(month)}_{band}.tif"
    shreklib.tif_export(np.round(aggregate/valid_count).astype(np.int16), shreklib.ref_grid["transform"], 32631,join(out_folder, outname), dtype="int16")

In [ ]:
out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/Sentinel2/Haute-Garonne/l3a_snapped"
snapped_fps=glob.glob(join(out_folder, "*.tif"))
fps_df=pd.DataFrame({"fps":snapped_fps})
parse_month=lambda fp:int(os.path.basename(fp)[:-4].split("_")[1][4:6])
parse_band=lambda fp:os.path.basename(fp)[:-4].split("_")[-1]
fps_df["band"]=fps_df.fps.apply(parse_band)
fps_df["month"]=fps_df.fps.apply(parse_month)
#on supprime les bandes qui ne sont pas des bandes optiques. Cela enlève toutes les bandes de traitement, comme les dates moyennes pondérées données par le cnes.
fps_df=fps_df.loc[fps_df.band.apply(lambda band:band[0]=="B")]
group=list(fps_df.groupby(by=["month", "band"]))[0]
shreklib.batch_map(mosaic, list(fps_df.groupby(by=["month", "band"])), num_workers=21)
means_process=[]
for group in list(fps_df.groupby(by=["month", "band"]))[:5]:
    means_process.append(mosaic(group))

# concaténation des bandes de même mois dans un seul geotif mensuel

In [ ]:
#fonction pour exporter un geotif issu de la concaténation des tifs spécifiés
def concat(grouped_object):
    id, group=grouped_object
    profile={
        "crs":32631,
        "transform":ref_transform,
        "height":ref_shape[0],
        "width":ref_shape[1],
        "dtype":"int16",
        "count":len(group),
        "interleave":"band",
        "tiled":True,
        "blockxsize":512,
        "blockysize":512
    }
    with rasterio.open(join(out_folder, f"{id}.tif"), "w", **profile) as dst:
        for i, (_, (fp, month)) in enumerate(group[["fps", "month"]].iterrows()):
            with rasterio.open(fp, "r") as src:
                dst.write(src.read(1), i+1)
                dst.set_band_description(i+1, month)

In [ ]:
month_band_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/Sentinel2/Haute-Garonne/month_bands_mosaics"
out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees_propres/S2"
month_band_df=pd.DataFrame({"fps":glob.glob(join(month_band_folder, "*.tif"))})
parse=lambda fp:os.path.basename(fp)[:-4].split("_")
month_band_df[["month", "band"]]=np.array(month_band_df.fps.apply(parse).tolist())
band_order=["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
month_order=["month1", "month2", "month3", "month4", "month5", "month6", "month7", "month8", "month9", "month10", "month11", "month12"]
month_band_df["band"]=pd.Categorical(month_band_df["band"], categories=band_order, ordered=True)
month_band_df["month"]=pd.Categorical(month_band_df["month"], categories=month_order, ordered=True)
month_band_df.sort_values(by=["band", "month"], inplace=True)
grouped=list(month_band_df.groupby(by="band"))[0]
for grouped in tqdm(list(month_band_df.groupby(by="band"))):
    concat(grouped)